# DeepGuard AI — Notebook 2 · Inference Pipeline

Contains every runtime file the app calls:

- `faces.py` — YOLO / YuNet face detection + video frame sampling + IoU tracker
- `predict.py` — 3-model ensemble, per-face quality-weighted voting, EMA smoothing, real Grad-CAM
- `forensics.py` — SHA-256, perceptual hash, EXIF, ffprobe, ELA
- `report.py` — reportlab-based PDF evidence report

## Runtime decision policy in `predict.py`

1. **Every detected face is scored independently**; a suspicious person is never averaged away by a group majority.
2. **Quality gates** use face area, detector confidence, blur, and brightness. Unsupported evidence returns `INCONCLUSIVE` without changing the raw probability.
3. **No suitable face** returns `NO_FACE`; file hashes and integrity checks remain available.
4. **Video is aggregated per tracked person** using repeated observations and a robust 75th percentile.
5. **Ateeqq AI-vs-human is excluded from the deepfake verdict** because synthetic-image detection is not equivalent to face-swap detection.
6. **Duplicate-face dHash is supporting metadata only** and cannot overwrite the classifier score.
7. **External challenge evaluation** is provided by `challenge_eval.py`; same-source validation AUC is not presented as forensic accuracy.


## `faces.py` — YOLO face detection + video sampling

In [ ]:
"""
Face detection using YOLOv8n-face (primary) with YuNet fallback.

Public API (contract unchanged):
    detect_and_crop(img_bgr, out_size=256, margin=0.28) -> list[np.ndarray]
    sample_video_frames(video_path, fps_sample=6.0) -> list[tuple[int, np.ndarray]]
    video_face_crops(video_path, ...) -> list[tuple[int, np.ndarray]]
    draw_face_box(img_bgr, box, color, thickness) -> np.ndarray

NEW additions:
    detect_with_boxes(img_bgr, out_size=256, margin=0.28)
        -> list[tuple[bbox_xywh, conf, crop_bgr]]
    video_face_crops_boxes(video_path, ...)
        -> list[tuple[frame_idx, bbox_xywh, conf, crop_bgr]]
"""
from __future__ import annotations
import os
import sys
from typing import Optional
import cv2
import numpy as np

# -------- YOLO primary --------
_YOLO_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)),
                          "models", "yolo", "yolov8n-face.pt")
_YOLO = None            # ultralytics.YOLO
_YOLO_LOAD_ERR = None

def _load_yolo():
    global _YOLO, _YOLO_LOAD_ERR
    if _YOLO is not None or _YOLO_LOAD_ERR is not None:
        return _YOLO
    try:
        from ultralytics import YOLO
        _YOLO = YOLO(_YOLO_PATH)
        # warmup on dummy
        _YOLO.predict(np.zeros((320, 320, 3), dtype=np.uint8), verbose=False)
    except Exception as e:
        _YOLO_LOAD_ERR = str(e)
        _YOLO = None
    return _YOLO


# -------- YuNet fallback --------
_YUNET_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)),
                           "face_detection_yunet_2023mar.onnx")
_YUNET_CACHE: dict[tuple[int, int], cv2.FaceDetectorYN] = {}

def _get_yunet(w: int, h: int):
    key = (w, h)
    if key not in _YUNET_CACHE and os.path.exists(_YUNET_PATH):
        _YUNET_CACHE[key] = cv2.FaceDetectorYN.create(
            model=_YUNET_PATH, config="", input_size=(w, h),
            score_threshold=0.5, nms_threshold=0.3, top_k=5000,
        )
    return _YUNET_CACHE.get(key)


# -------- shared crop helper --------
def _crop_with_margin(img: np.ndarray, xywh, margin: float, out_size: int) -> np.ndarray:
    x, y, w, h = [int(v) for v in xywh]
    cx = x + w / 2.0; cy = y + h / 2.0
    side = max(w, h) * (1.0 + 2.0 * margin)
    x1 = int(round(cx - side / 2.0)); y1 = int(round(cy - side / 2.0))
    x2 = int(round(cx + side / 2.0)); y2 = int(round(cy + side / 2.0))
    x1 = max(0, x1); y1 = max(0, y1)
    x2 = min(img.shape[1], x2); y2 = min(img.shape[0], y2)
    if x2 <= x1 or y2 <= y1:
        return cv2.resize(img, (out_size, out_size))
    return cv2.resize(img[y1:y2, x1:x2], (out_size, out_size))


# -------- Detection: unified interface --------
def _detect_yolo(img_bgr: np.ndarray, conf_threshold: float = 0.25):
    """Return list of (x, y, w, h, conf). Empty if YOLO fails or no faces."""
    yolo = _load_yolo()
    if yolo is None:
        return []
    try:
        res = yolo.predict(img_bgr, conf=conf_threshold, verbose=False, device=0)
        out = []
        for r in res:
            boxes = r.boxes
            if boxes is None or boxes.xyxy is None: continue
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            for (x1, y1, x2, y2), c in zip(xyxy, confs):
                out.append((int(x1), int(y1), int(x2 - x1), int(y2 - y1), float(c)))
        return out
    except Exception:
        return []


def _detect_yunet(img_bgr: np.ndarray):
    """Return list of (x, y, w, h, conf)."""
    h_img, w_img = img_bgr.shape[:2]
    det = _get_yunet(w_img, h_img)
    if det is None: return []
    _, faces = det.detect(img_bgr)
    if faces is None: return []
    out = []
    for f in faces:
        x, y, w, h = [int(v) for v in f[0:4]]
        conf = float(f[-1]) if len(f) > 4 else 0.9
        out.append((x, y, w, h, conf))
    return out


def _detect_any(img_bgr: np.ndarray):
    """YOLO first; YuNet fallback."""
    hits = _detect_yolo(img_bgr)
    if hits: return hits
    return _detect_yunet(img_bgr)


# ========================== PUBLIC API =====================================

def detect_and_crop(img_bgr: np.ndarray, out_size: int = 256,
                    margin: float = 0.28) -> list[np.ndarray]:
    """Return 256x256 BGR uint8 face crops. If none, returns [whole-image resized]."""
    hits = _detect_any(img_bgr)
    if not hits:
        return [cv2.resize(img_bgr, (out_size, out_size))]
    return [_crop_with_margin(img_bgr, h[:4], margin, out_size) for h in hits]


def detect_with_boxes(img_bgr: np.ndarray, out_size: int = 256,
                      margin: float = 0.28
                      ) -> list[tuple[tuple[int, int, int, int], float, np.ndarray]]:
    """Return list of ((x,y,w,h), confidence, crop_256x256). Empty if no faces found."""
    hits = _detect_any(img_bgr)
    out = []
    for x, y, w, h, conf in hits:
        crop = _crop_with_margin(img_bgr, (x, y, w, h), margin, out_size)
        out.append(((x, y, w, h), conf, crop))
    return out


def sample_video_frames(video_path: str,
                        fps_sample: float = 6.0
                        ) -> list[tuple[int, np.ndarray]]:
    """Sample frames at target fps (default 6). Handles very short videos."""
    cap = cv2.VideoCapture(video_path)
    frames: list[tuple[int, np.ndarray]] = []
    try:
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0 or np.isnan(fps): fps = 30.0
        step = max(1, int(round(fps / fps_sample)))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        if 0 < total < step:
            mid = total // 2
            cap.set(cv2.CAP_PROP_POS_FRAMES, mid)
            ok, frame = cap.read()
            if ok and frame is not None:
                return [(mid, frame)]
        idx = 0
        while True:
            ok, frame = cap.read()
            if not ok: break
            if idx % step == 0 and frame is not None:
                frames.append((idx, frame))
            idx += 1
    finally:
        cap.release()
    return frames


def video_face_crops(video_path: str, fps_sample: float = 6.0,
                     out_size: int = 256, margin: float = 0.28
                     ) -> list[tuple[int, np.ndarray]]:
    """Every face in every sampled frame. Returns (frame_idx, crop)."""
    pairs = sample_video_frames(video_path, fps_sample)
    out = []
    for idx, frame in pairs:
        hits = _detect_any(frame)
        if not hits:
            out.append((idx, cv2.resize(frame, (out_size, out_size))))
            continue
        for x, y, w, h, _ in hits:
            out.append((idx, _crop_with_margin(frame, (x, y, w, h), margin, out_size)))
    return out


def video_face_crops_boxes(video_path: str, fps_sample: float = 6.0,
                           out_size: int = 256, margin: float = 0.28
                           ) -> list[tuple[int, tuple[int, int, int, int], float, np.ndarray]]:
    """Every face in every sampled frame WITH bboxes. For tracker + UI overlay."""
    pairs = sample_video_frames(video_path, fps_sample)
    out = []
    for idx, frame in pairs:
        hits = _detect_any(frame)
        if not hits:
            h_img, w_img = frame.shape[:2]
            out.append((idx, (0, 0, w_img, h_img), 0.0,
                        cv2.resize(frame, (out_size, out_size))))
            continue
        for x, y, w, h, conf in hits:
            crop = _crop_with_margin(frame, (x, y, w, h), margin, out_size)
            out.append((idx, (x, y, w, h), conf, crop))
    return out


def draw_face_box(img_bgr: np.ndarray, box,
                  color=(0, 0, 255), thickness: int = 3) -> np.ndarray:
    x, y, w, h = [int(v) for v in box[:4]]
    out = img_bgr.copy()
    cv2.rectangle(out, (x, y), (x + w, y + h), color, thickness)
    return out


def draw_face_boxes(img_bgr: np.ndarray,
                    items: list[tuple[tuple[int, int, int, int], float, str]],
                    ) -> np.ndarray:
    """Draw multiple face boxes with labels. items: (bbox, score, label)."""
    out = img_bgr.copy()
    for bbox, score, label in items:
        x, y, w, h = [int(v) for v in bbox[:4]]
        color = (0, 0, 255) if score >= 0.5 else (0, 255, 0)
        cv2.rectangle(out, (x, y), (x + w, y + h), color, 3)
        text = f"{label} {score*100:.0f}%"
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(out, (x, y - th - 8), (x + tw + 6, y), color, -1)
        cv2.putText(out, text, (x + 3, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return out


# ========================== CLI demo =======================================
if __name__ == "__main__":
    if len(sys.argv) < 2:
        print(f"Usage: {sys.argv[0]} <image_or_video_path>"); sys.exit(1)
    path = sys.argv[1]
    ext = os.path.splitext(path)[1].lower()
    if ext in [".mp4", ".mov", ".avi", ".mkv"]:
        crops = video_face_crops_boxes(path)
        print(f"Sampled {len(set(c[0] for c in crops))} frames, "
              f"total {len(crops)} face crops")
        if crops:
            cv2.imwrite("crop_first.jpg", crops[0][3])
            cv2.imwrite("crop_last.jpg",  crops[-1][3])
    else:
        img = cv2.imread(path)
        if img is None:
            print(f"could not read {path}"); sys.exit(1)
        results = detect_with_boxes(img)
        print(f"Found {len(results)} face(s) using "
              f"{'YOLO' if _load_yolo() else 'YuNet'}")
        for i, (bbox, conf, crop) in enumerate(results):
            print(f"  face {i}: bbox={bbox}  conf={conf:.3f}")
            cv2.imwrite(f"crop_{i}.jpg", crop)


## `predict.py` — per-face/per-track inference, quality gating, Grad-CAM

In [ ]:
"""
Inference wrapper — the ONLY module the UI (app.py) imports for verdicts.

Exposes:
    predict(file_path) -> dict     # branches on extension: image vs video
    predict_image(path) -> dict
    predict_video(path, fps_sample=2.0) -> dict

Uses:
    faces.py       for face detection + video sampling (Track 2 teammate)
    model_best.pt  for the trained classifier weights (my training run)

Label convention:
    output score in [0, 1] = probability the input is FAKE
    verdict = "FAKE" if score > 0.5 else "REAL"
"""
from __future__ import annotations
import io, os, time
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import cv2
from PIL import Image
from torchvision import transforms, models

try:
    from faces import detect_and_crop, video_face_crops
    try:
        from faces import detect_with_boxes, video_face_crops_boxes
        _HAS_BOX_API = True
    except ImportError:
        _HAS_BOX_API = False
except Exception as e:
    raise ImportError(
        "predict.py requires faces.py from Track 2 to be in the same folder"
    ) from e


# ==================== Simple IoU tracker + EMA ====================

def _iou(a, b) -> float:
    """IoU of two boxes in xywh format."""
    ax, ay, aw, ah = a; bx, by, bw, bh = b
    x1 = max(ax, bx); y1 = max(ay, by)
    x2 = min(ax + aw, bx + bw); y2 = min(ay + ah, by + bh)
    if x2 <= x1 or y2 <= y1: return 0.0
    inter = (x2 - x1) * (y2 - y1)
    union = aw * ah + bw * bh - inter
    return inter / max(union, 1.0)


class IoUTracker:
    """
    Frame-by-frame IoU tracker with a fixed disappear budget.
    Assigns persistent integer face_ids to overlapping bboxes across frames.
    """
    def __init__(self, iou_threshold: float = 0.3, max_missed: int = 8):
        self.iou_threshold = iou_threshold
        self.max_missed = max_missed
        self.next_id = 0
        # id -> {"bbox": xywh, "last_seen": frame_idx, "missed": int}
        self.tracks: dict[int, dict] = {}

    def update(self, frame_idx: int,
               detections: list[tuple[int, int, int, int]]
               ) -> list[int]:
        """Assign a face_id to each detection. Returns list of face_ids parallel to detections."""
        # 1. Match detections to existing tracks by best-IoU
        assignments: list[Optional[int]] = [None] * len(detections)
        used_track_ids: set[int] = set()
        # Greedy match: for each detection, find best-matching unused track
        for det_i, det in enumerate(detections):
            best_id, best_iou = None, self.iou_threshold
            for tid, track in self.tracks.items():
                if tid in used_track_ids: continue
                score = _iou(det, track["bbox"])
                if score > best_iou:
                    best_iou = score; best_id = tid
            if best_id is not None:
                assignments[det_i] = best_id
                used_track_ids.add(best_id)
        # 2. New tracks for unmatched detections
        for det_i, det in enumerate(detections):
            if assignments[det_i] is None:
                new_id = self.next_id; self.next_id += 1
                self.tracks[new_id] = {"bbox": det, "last_seen": frame_idx, "missed": 0}
                assignments[det_i] = new_id
            else:
                tid = assignments[det_i]
                self.tracks[tid]["bbox"] = det
                self.tracks[tid]["last_seen"] = frame_idx
                self.tracks[tid]["missed"] = 0
        # 3. Age & prune tracks that were not matched this frame
        for tid, track in list(self.tracks.items()):
            if track["last_seen"] != frame_idx:
                track["missed"] += 1
                if track["missed"] > self.max_missed:
                    del self.tracks[tid]
        return assignments  # type: ignore


class EMASmoother:
    """Exponential moving average per key. alpha computed from window N."""
    def __init__(self, window: int = 15):
        self.alpha = 2.0 / (window + 1)
        self.state: dict[int, float] = {}
    def update(self, key: int, value: float) -> float:
        prev = self.state.get(key, value)  # first observation is not smoothed
        smoothed = self.alpha * value + (1 - self.alpha) * prev
        self.state[key] = smoothed
        return smoothed
    def peek(self, key: int) -> Optional[float]:
        return self.state.get(key)

# ---- config ----
# Ensemble: v1 (Hemg-only) + v2 (Hemg + SD + aug). Take max score.
# Rationale: v1 is more suspicious, v2 handles OOD real photos.
# Also analyze BOTH tight face + wider context (face-swap artifacts live at boundary)
WEIGHTS_PATH   = os.environ.get("MODEL_WEIGHTS",   "model_v3_best.pt")
WEIGHTS_PATH_2 = os.environ.get("MODEL_WEIGHTS_2", "model_v2_best.pt")
IMG_SIZE       = 224
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME     = "Ensemble: EfficientNet-B0 v3+v2 (CIFAKE+SD+aug) + Ateeqq-ViT"
MODEL_VERSION  = "5.0-tri-ensemble-cifake"
ATEEQQ_PATH    = r"C:\dl\deepfake\models\ateeqq_aivshuman"
ATEEQQ_WEIGHT  = 0.0        # AI-generated-image detector excluded from deepfake verdict
LOCAL_WEIGHT   = 1.0        # our own models keep full weight
FPS_SAMPLE     = 4.0    # more frames per second (was 2) — better video coverage
TOP_K_FRAMES   = 5
MIN_FRAMES     = 8      # if video is short, sample at least this many frames
FAKE_THRESHOLD = 0.35   # calibrated: catches SD/StyleGAN without flooding real photos
WIDE_MARGIN = 0.9
MIN_FACE_AREA_FRAC = 0.005
MIN_DETECTION_CONF = 0.25
MIN_RELIABLE_BLUR = 50.0
MIN_TRACK_OBSERVATIONS = 3
MIN_RELIABLE_BLUR = 50.0

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

_preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ---- lazy ensemble of models ----
_MODELS: list[nn.Module] = []
_MODEL_META: dict = {}
_ATEEQQ_PROC = None
_ATEEQQ_MODEL = None
_ATEEQQ_AI_IDX = 0


def _build_arch() -> nn.Module:
    m = models.efficientnet_b0(weights=None)
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=True), nn.Linear(in_f, 1))
    return m


def _load_one(path: str) -> Optional[nn.Module]:
    if not os.path.exists(path): return None
    ckpt = torch.load(path, map_location=DEVICE)
    m = _build_arch()
    m.load_state_dict(ckpt["model"])
    m.to(DEVICE).eval()
    return m, ckpt


def _load_models():
    global _MODELS, _MODEL_META
    if _MODELS: return _MODELS
    loaded = []
    metas = []
    for p in (WEIGHTS_PATH, WEIGHTS_PATH_2):
        r = _load_one(p)
        if r is not None:
            m, ckpt = r
            loaded.append(m)
            metas.append({"path": p, "val_auc": ckpt.get("val_auc"),
                          "version": ckpt.get("version", "?")})
    if not loaded:
        raise FileNotFoundError(f"No model weights found ({WEIGHTS_PATH}, {WEIGHTS_PATH_2})")
    _MODELS = loaded
    _MODEL_META = {"models": metas, "n_models": len(loaded),
                   "threshold": FAKE_THRESHOLD}
    return _MODELS


# back-compat single-model accessor
def _load_model() -> nn.Module:
    return _load_models()[0]


# --------------- core inference ---------------

def _bgr_to_tensor(bgr: np.ndarray) -> torch.Tensor:
    """224x224 BGR uint8 -> 1x3x224x224 float, ImageNet-normalized."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    return _preprocess(pil)


def _load_ateeqq():
    """Load Ateeqq's HuggingFace AI-vs-Human ViT model."""
    global _ATEEQQ_PROC, _ATEEQQ_MODEL, _ATEEQQ_AI_IDX
    if _ATEEQQ_MODEL is not None:
        return _ATEEQQ_PROC, _ATEEQQ_MODEL, _ATEEQQ_AI_IDX
    if not os.path.exists(ATEEQQ_PATH):
        return None, None, 0
    try:
        from transformers import AutoImageProcessor, AutoModelForImageClassification
        _ATEEQQ_PROC  = AutoImageProcessor.from_pretrained(ATEEQQ_PATH)
        _ATEEQQ_MODEL = AutoModelForImageClassification.from_pretrained(ATEEQQ_PATH).to(DEVICE).eval()
        for i, l in _ATEEQQ_MODEL.config.id2label.items():
            if any(k in l.lower() for k in ("ai", "fake", "gen", "syn")):
                _ATEEQQ_AI_IDX = i; break
    except Exception as e:
        print(f"[predict] Ateeqq load failed, continuing without: {e}")
        _ATEEQQ_MODEL = None
    return _ATEEQQ_PROC, _ATEEQQ_MODEL, _ATEEQQ_AI_IDX


@torch.no_grad()
def _score_batch(crops_bgr: list[np.ndarray]) -> np.ndarray:
    """
    Tri-model ensemble: v1 + v2 EfficientNets (MAX between them) then
    WEIGHTED MEAN with Ateeqq ViT.
        combined = (LOCAL_WEIGHT * max(v1,v2) + ATEEQQ_WEIGHT * ateeqq) /
                   (LOCAL_WEIGHT + ATEEQQ_WEIGHT)
    Cancels individual model biases while keeping deepfake sensitivity.
    """
    if not crops_bgr:
        return np.array([], dtype=np.float32)
    ms = _load_models()
    batch = torch.stack([_bgr_to_tensor(c) for c in crops_bgr]).to(DEVICE)

    # Our two local models
    local_scores = []
    for m in ms:
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            logits = m(batch).squeeze(1)
        local_scores.append(torch.sigmoid(logits).float().cpu().numpy())
    local_max = np.stack(local_scores, axis=0).max(axis=0)   # worst-of-local

    # Ateeqq ViT (optional; skips gracefully if not present)
    if ATEEQQ_WEIGHT <= 0:
        return local_max
    proc, ateeqq, ai_idx = _load_ateeqq()
    if ateeqq is None:
        return local_max
    try:
        pil_imgs = [Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c in crops_bgr]
        x = proc(images=pil_imgs, return_tensors="pt").to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            logits = ateeqq(**x).logits
        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
        ateeqq_scores = probs[:, ai_idx]
    except Exception:
        return local_max

    combined = (LOCAL_WEIGHT * local_max + ATEEQQ_WEIGHT * ateeqq_scores) / (LOCAL_WEIGHT + ATEEQQ_WEIGHT)
    return combined


def _verdict_from_score(score: float, band: float = 0.05) -> tuple[str, float]:
    """
    Return (verdict, confidence).
    Uses FAKE_THRESHOLD (default 0.35) — more sensitive than 0.5 to catch
    face-swap deepfakes the models are under-confident on.
    """
    thr = FAKE_THRESHOLD
    if abs(score - thr) < band:
        return "INCONCLUSIVE", 0.0
    if score > thr:
        return "FAKE", float(min(1.0, (score - thr) / max(1 - thr, 1e-6)))
    return "REAL", float(min(1.0, (thr - score) / max(thr, 1e-6)))


_GRADCAM = None

def _get_gradcam():
    """Lazy Grad-CAM instance targeting last conv block of Model 1."""
    global _GRADCAM
    if _GRADCAM is not None: return _GRADCAM
    try:
        from pytorch_grad_cam import GradCAM
        ms = _load_models()
        m = ms[0]
        # For EfficientNet-B0: features[-1] is the final conv layer (Conv-BN-SiLU stack)
        target_layers = [m.features[-1]]
        _GRADCAM = GradCAM(model=m, target_layers=target_layers)
    except Exception as e:
        print(f"[predict] Grad-CAM disabled: {e}")
        _GRADCAM = False
    return _GRADCAM


def _real_gradcam_heatmap(bgr: np.ndarray) -> np.ndarray:
    """Real Grad-CAM overlay on the image, showing where the model looked."""
    cam = _get_gradcam()
    if cam is False:
        return _fallback_heatmap(bgr)
    try:
        from pytorch_grad_cam.utils.image import show_cam_on_image
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        pil_shape = (IMG_SIZE, IMG_SIZE)
        rgb_resized = cv2.resize(rgb, pil_shape).astype(np.float32) / 255.0
        input_tensor = _bgr_to_tensor(cv2.resize(bgr, pil_shape)).unsqueeze(0).to(DEVICE)
        # Target: increase FAKE probability (single-logit BCE model → target 0)
        with torch.enable_grad():
            grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0]
        visualization = show_cam_on_image(rgb_resized, grayscale_cam,
                                          use_rgb=True, image_weight=0.55)
        # Convert RGB back to BGR to match the rest of the pipeline
        overlay_bgr = cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR)
        # Free the extra gradient memory
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        # Resize back to the original crop size
        return cv2.resize(overlay_bgr, (bgr.shape[1], bgr.shape[0]))
    except Exception as e:
        print(f"[predict] Grad-CAM inference failed, fallback: {e}")
        return _fallback_heatmap(bgr)


def _fallback_heatmap(bgr: np.ndarray) -> np.ndarray:
    """Non-Grad-CAM fallback if the library fails (edges + jet colormap)."""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Sobel(gray, cv2.CV_32F, 1, 1, ksize=5)
    edges = cv2.GaussianBlur(np.abs(edges), (0, 0), sigmaX=8)
    edges = cv2.normalize(edges, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    heat = cv2.applyColorMap(edges, cv2.COLORMAP_JET)
    return cv2.addWeighted(bgr, 0.55, heat, 0.45, 0)


def _fake_heatmap(bgr: np.ndarray) -> np.ndarray:
    """Backwards-compat name — now runs REAL Grad-CAM."""
    return _real_gradcam_heatmap(bgr)


# --------------- public API ---------------

def predict_image(path: str) -> dict:
    """
    QUALITY-WEIGHTED per-face voting:
      - Detect ALL faces WITH bboxes + YOLO confidence
      - Filter out tiny background faces (< 3% of frame area)
      - Score each face at tight + wide crops (worst-of-two per face)
      - Weight each face's contribution by (area × detection_confidence)
      - Final verdict = weighted MEAN of face scores  (not naive MAX)
      - PRIMARY face for display = largest area (subject of the photo)
    A blurry background face no longer flips the verdict of a large clear face.
    """
    t0 = time.time()
    bgr = cv2.imread(path)
    if bgr is None:
        raise ValueError(f"could not read image: {path}")
    H, W = bgr.shape[:2]
    frame_area = H * W

    # Get faces WITH boxes + confidence for quality weighting
    if _HAS_BOX_API:
        detections = detect_with_boxes(bgr, out_size=IMG_SIZE, margin=0.3)  # [(bbox,conf,crop)]
    else:
        legacy = detect_and_crop(bgr, out_size=IMG_SIZE, margin=0.3)
        detections = [((0, 0, W, H), 0.9, c) for c in legacy]

    # Filter: keep faces at least 3% of frame area
    MIN_AREA_FRAC = 0.03
    kept = []
    for bbox, conf, crop in detections:
        _, _, w, h = bbox
        area_frac = (w * h) / max(frame_area, 1)
        if area_frac >= MIN_FACE_AREA_FRAC and conf >= MIN_DETECTION_CONF:
            kept.append((bbox, conf, crop, area_frac))


    # Wide crops for boundary artifact analysis (same faces, wider margin)
    tight_crops = [k[2] for k in kept]
    wide_crops  = []
    if kept:
        for bbox, _, _, _ in kept:
            wide_crops.append(_crop_wide(bgr, bbox, WIDE_MARGIN, IMG_SIZE))
    full_crop = cv2.resize(bgr, (IMG_SIZE, IMG_SIZE))

    all_crops = tight_crops + wide_crops + [full_crop]
    scores = _score_batch(all_crops) if all_crops else np.array([0.5])
    n_f = len(tight_crops)

    per_face_scores, weights, blur_scores = [], [], []
    tight_scores, wide_scores, scale_labels = [], [], []
    for i, (bbox, det_conf, crop_i, area_frac) in enumerate(kept):
        s_tight = float(scores[i])
        s_wide  = float(scores[n_f + i]) if i < n_f else s_tight
        # Require agreement between the tight face crop and wider context. The
        # old max() rule turned crop-specific artifacts into extreme false positives.
        face_score = float(np.sqrt(max(s_tight, 0.0) * max(s_wide, 0.0)))
        tight_scores.append(s_tight); wide_scores.append(s_wide)
        if s_tight > FAKE_THRESHOLD and s_wide > FAKE_THRESHOLD:
            scale_labels.append("FAKE")
        elif s_tight < FAKE_THRESHOLD and s_wide < FAKE_THRESHOLD:
            scale_labels.append("REAL")
        else:
            scale_labels.append("INCONCLUSIVE")
        per_face_scores.append(face_score)
        # NEW: blur factor — blurry face gets down-weighted (model can't judge it)
        blur_w = _blur_weight(crop_i)
        blur_scores.append(_blur_score(crop_i))
        # Weight = area × detection confidence × sharpness
        weights.append(area_frac * det_conf * blur_w)
    s_full = float(scores[-1]) if len(scores) > 2 * n_f else 0.5

    # Forensic group rule: one reliable suspicious face must not be hidden by
    # a median/majority of authentic faces.
    reliable_indices = [
        i for i, b in enumerate(blur_scores)
        if b >= MIN_RELIABLE_BLUR
        and kept[i][1] >= MIN_DETECTION_CONF
        and kept[i][3] >= MIN_FACE_AREA_FRAC
    ]
    if per_face_scores:
        candidates = reliable_indices or list(range(len(per_face_scores)))
        weighted_face_score = float(max(per_face_scores[i] for i in candidates))
        # Primary face = LARGEST detected face (main subject)
        primary_idx = int(np.argmax([k[3] for k in kept]))
        primary_crop = kept[primary_idx][2]
    else:
        weighted_face_score = s_full
        primary_idx = 0
        primary_crop = full_crop

    # Face-level evidence drives the verdict; full-frame score is metadata only.
    score = weighted_face_score

    # ═══ Copy-paste face tampering check — ONLY for exactly 2 faces (Mona Lisa) ═══
    dup_sim = 0.0; dup_i = dup_j = -1
    if len(kept) == 2:
        dup_sim, dup_i, dup_j = _detect_duplicate_faces([k[2] for k in kept])


    # Quality gating: do not mutate the probability to manufacture uncertainty.
    avg_brightness = float(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).mean())
    has_reliable_face = bool(reliable_indices) and avg_brightness >= 45
    verdict, conf = _verdict_from_score(score)
    reliable_scale_labels = [scale_labels[i] for i in reliable_indices]
    if not detections:
        verdict, conf = "NO_FACE", 0.0
    elif not has_reliable_face:
        verdict, conf = "INCONCLUSIVE", 0.0
    elif "FAKE" in reliable_scale_labels:
        verdict, conf = _verdict_from_score(score)
    elif reliable_scale_labels and all(v == "REAL" for v in reliable_scale_labels):
        verdict, conf = "REAL", max(conf, 0.5)
    else:
        verdict, conf = "INCONCLUSIVE", 0.0
    # Build all-face list for UI (grid display)
    all_faces_data = []
    for i, (bbox, det_conf, crop, area_frac) in enumerate(kept):
        reliable = i in reliable_indices
        face_verdict = scale_labels[i]
        face_conf = (_verdict_from_score(per_face_scores[i])[1]
                     if face_verdict != "INCONCLUSIVE" else 0.0)
        if not reliable:
            face_verdict, face_conf = "INCONCLUSIVE", 0.0
        all_faces_data.append({
            "index": i,
            "score": float(per_face_scores[i]) if i < len(per_face_scores) else 0.0,
            "weight": float(weights[i]) if i < len(weights) else 0.0,
            "blur": float(blur_scores[i]) if i < len(blur_scores) else 0.0,
            "area_frac": float(area_frac),
            "det_conf": float(det_conf),
            "bbox": bbox,
            "crop_bgr": crop,
            "heatmap_bgr": _fake_heatmap(crop),
            "is_primary": i == primary_idx,
            "reliable": reliable,
            "verdict": face_verdict,
            "confidence": face_conf,
        })

    _load_models()
    return {
        "kind": "image",
        "score": score,
        "verdict": verdict,
        "confidence": conf,
        "face_crop_bgr": primary_crop,
        "heatmap_bgr": _fake_heatmap(primary_crop),
        "all_faces": all_faces_data,               # NEW: full list for UI grid
        "elapsed_ms": (time.time() - t0) * 1000,
        "model_name": MODEL_NAME,
        "model_version": MODEL_VERSION,
        "meta": {**_MODEL_META,
                 "n_faces_detected": len(detections),
                 "n_faces_kept": len(kept),
                 "per_face_scores": [round(s, 4) for s in per_face_scores],
                 "tight_face_scores": [round(s, 4) for s in tight_scores],
                 "wide_face_scores": [round(s, 4) for s in wide_scores],
                 "scale_labels": scale_labels,
                 "per_face_weights": [round(w, 4) for w in weights],
                 "weighted_face_score": round(weighted_face_score, 4),
                 "full_frame_score": round(s_full, 4),
                 "primary_face_idx": primary_idx,
                 "duplicate_face_similarity": round(dup_sim, 3),
                 "duplicate_face_pair": [dup_i, dup_j],
                 "per_face_blur_scores": [round(b, 1) for b in blur_scores]},
        "quality": {
            "reliable": has_reliable_face,
            "brightness": round(avg_brightness, 1),
            "reason": None if has_reliable_face else (
                "no face detected" if not detections else
                "detected face is too small, blurry, dark, or low-confidence"
            ),
        },
    }


def _blur_score(bgr: np.ndarray) -> float:
    """
    Laplacian variance = sharpness. Higher = sharper.
    Real photos: usually > 100. Motion-blurred / out-of-focus: < 50.
    Model can't reliably judge blurred faces — we down-weight them.
    """
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def _blur_weight(bgr: np.ndarray) -> float:
    """
    Convert sharpness → weight multiplier in [0.15, 1.0].
    Very blurry face contributes only 15%; sharp face full weight.
    """
    v = _blur_score(bgr)
    # sigmoid-ish: <30 = 0.15, 60 = 0.5, >120 = 1.0
    if v < 30: return 0.15
    if v < 60: return 0.35
    if v < 100: return 0.7
    return 1.0


def _dhash_face(bgr: np.ndarray, hash_size: int = 16) -> np.ndarray:
    """Difference hash of a face crop, as a bit array."""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (hash_size + 1, hash_size))
    return (resized[:, 1:] > resized[:, :-1]).flatten()


def _detect_duplicate_faces(crops: list[np.ndarray]) -> tuple[float, int, int]:
    """
    Check if any two face crops in an image are near-identical (copy-paste
    tampering, e.g. cloned face in the Mona Lisa case).
    Returns (max_similarity, i, j) where i, j are the matching face indices.
    Similarity > 0.9 = almost certainly a copy-paste.
    """
    if len(crops) < 2:
        return 0.0, -1, -1
    hashes = [_dhash_face(c) for c in crops]
    max_sim = 0.0; best = (-1, -1)
    for i in range(len(hashes)):
        for j in range(i + 1, len(hashes)):
            sim = float((hashes[i] == hashes[j]).mean())
            if sim > max_sim:
                max_sim = sim; best = (i, j)
    return max_sim, best[0], best[1]


def _crop_wide(img: np.ndarray, xywh, margin: float, out_size: int) -> np.ndarray:
    """Same crop math as faces.py but inline (avoids importing internal helper)."""
    x, y, w, h = [int(v) for v in xywh]
    cx, cy = x + w / 2, y + h / 2
    side = max(w, h) * (1 + 2 * margin)
    x1 = int(max(0, cx - side / 2)); y1 = int(max(0, cy - side / 2))
    x2 = int(min(img.shape[1], cx + side / 2)); y2 = int(min(img.shape[0], cy + side / 2))
    if x2 <= x1 or y2 <= y1:
        return cv2.resize(img, (out_size, out_size))
    return cv2.resize(img[y1:y2, x1:x2], (out_size, out_size))


def predict_video(path: str, fps_sample: float = FPS_SAMPLE) -> dict:
    """
    Multi-face + TRACKED video pipeline (v3):
      - Sample video at fps_sample (default 6 fps ~ every 5th frame @ 30fps)
      - For every sampled frame, detect ALL faces with bboxes (YOLO)
      - IoU-track faces across frames → assign persistent face_id
      - Score every face crop through ensemble
      - Smooth each face_id's score with EMA over a 15-frame window
      - Per-frame score = MAX(smoothed scores across faces in that frame)
      - Video score     = mean(top-25% per-frame scores)  (skews to suspicious)
    Returns extra `per_face_timeline` for UI: score curves per tracked face.
    """
    from collections import defaultdict
    t0 = time.time()

    # -------- Sample + detect (with bboxes if faces.py provides them) --------
    used_fps = fps_sample
    if _HAS_BOX_API:
        pairs = video_face_crops_boxes(path, fps_sample=fps_sample,
                                       out_size=IMG_SIZE, margin=0.3)
        if len({p[0] for p in pairs}) < MIN_FRAMES:
            for try_fps in (8.0, 15.0, 30.0):
                if try_fps <= used_fps: continue
                pairs = video_face_crops_boxes(path, fps_sample=try_fps,
                                               out_size=IMG_SIZE, margin=0.3)
                used_fps = try_fps
                if len({p[0] for p in pairs}) >= MIN_FRAMES: break
    else:
        # Fallback: legacy signature without bboxes → assign dummy bbox 0,0,0,0
        legacy = video_face_crops(path, fps_sample=fps_sample,
                                  out_size=IMG_SIZE, margin=0.3)
        pairs = [(idx, (0, 0, 0, 0), 1.0, crop) for idx, crop in legacy]
    fps_sample = used_fps
    if not pairs:
        raise ValueError(f"no frames extracted from {path}")

    # -------- Score every face crop through ensemble (batched) --------
    crops = [p[3] for p in pairs]
    CHUNK = 32   # smaller chunk for Grad-CAM VRAM budget
    scored = []
    for i in range(0, len(crops), CHUNK):
        scored.append(_score_batch(crops[i:i + CHUNK]))
    raw_scores = np.concatenate(scored) if scored else np.array([], dtype=np.float32)

    # -------- Group by frame, run tracker + EMA in temporal order --------
    # ALSO: filter tiny background faces (< 3% of frame area) — same fix as images
    MIN_AREA_FRAC = 0.03
    frames_dict: dict[int, list[tuple[tuple, float, float, float, np.ndarray]]] = defaultdict(list)
    # tuple: (bbox, det_conf, area_frac, score, crop)
    for (frame_idx, bbox, det_conf, crop), s in zip(pairs, raw_scores):
        _, _, w, h = bbox
        # Frame dims from crop.shape isn't right (crop is 224). Use bbox pixels vs whole video frame guess: fall back to conservative area check via bbox pixels only
        area_pixels = w * h
        # Keep detections above ~10k pixels absolute (roughly 100×100 face) OR pass; we don't have frame dims here
        # We'll do relative filter later inside per-frame loop
        frames_dict[frame_idx].append((bbox, float(det_conf), area_pixels, float(s), crop))
    frame_indices = sorted(frames_dict.keys())

    tracker = IoUTracker(iou_threshold=0.3, max_missed=8)
    ema     = EMASmoother(window=15)
    per_face_timeline: dict[int, list[tuple[int, float, float]]] = defaultdict(list)
    reliable_track_scores: dict[int, list[float]] = defaultdict(list)
    per_frame_score: dict[int, float] = {}
    per_frame_best_crop: dict[int, np.ndarray] = {}
    per_frame_best_bbox: dict[int, tuple] = {}
    per_frame_n_faces: dict[int, int] = {}

    for frame_idx in frame_indices:
        faces_in_frame = frames_dict[frame_idx]
        # Frame-relative area filter: keep faces at least 30% of the biggest face in this frame
        max_area = max((f[2] for f in faces_in_frame), default=1)
        kept = [f for f in faces_in_frame if f[2] >= max(0.3 * max_area, 5000)]
        if not kept: kept = faces_in_frame  # fallback

        bboxes = [f[0] for f in kept]
        face_ids = tracker.update(frame_idx, bboxes)

        # Weighted aggregate for this frame
        weighted_num = 0.0; weighted_den = 0.0
        reliable_frame_scores = []
        best_crop = None; best_bbox = None; largest_area = -1
        for (bbox, det_conf, area, raw_s, crop), fid in zip(kept, face_ids):
            smoothed = ema.update(fid, raw_s)
            per_face_timeline[fid].append((frame_idx, float(smoothed), float(raw_s)))
            if det_conf >= MIN_DETECTION_CONF and area >= 5000 and _blur_score(crop) >= MIN_RELIABLE_BLUR:
                reliable_frame_scores.append(float(smoothed))
                reliable_track_scores[fid].append(float(smoothed))
            weight = area * det_conf
            weighted_num += smoothed * weight
            weighted_den += weight
            if area > largest_area:
                largest_area = area; best_crop = crop; best_bbox = bbox
        frame_score = max(reliable_frame_scores) if reliable_frame_scores else 0.5
        per_frame_score[frame_idx]     = float(frame_score)
        per_frame_best_crop[frame_idx] = best_crop if best_crop is not None else crops[0]
        per_frame_best_bbox[frame_idx] = best_bbox if best_bbox is not None else (0, 0, 0, 0)
        per_frame_n_faces[frame_idx]   = len(kept)

    per_frame     = [per_frame_score[i] for i in frame_indices]
    per_frame_arr = np.array(per_frame, dtype=np.float32)

    # Aggregate each person independently, then preserve the most suspicious
    # reliable track. This prevents other people from averaging it away.
    track_summaries = {}
    for fid, values in reliable_track_scores.items():
        if len(values) >= MIN_TRACK_OBSERVATIONS:
            track_summaries[fid] = float(np.percentile(values, 75))
    track_results = []
    for fid, score_i in sorted(track_summaries.items()):
        verdict_i, confidence_i = _verdict_from_score(score_i)
        track_results.append({"face_id": int(fid), "score": score_i,
                              "verdict": verdict_i,
                              "confidence": confidence_i,
                              "observations": len(reliable_track_scores[fid])})
    if track_summaries:
        agg = max(track_summaries.values())
        verdict, conf = _verdict_from_score(agg)
    else:
        agg = 0.5
        has_face_detection = any(float(p[2]) >= MIN_DETECTION_CONF for p in pairs)
        verdict, conf = (("INCONCLUSIVE", 0.0) if has_face_detection
                         else ("NO_FACE", 0.0))

    # -------- Top-K most suspicious frames --------
    order = np.argsort(-per_frame_arr)[:TOP_K_FRAMES]
    top_frames = []
    for i in order:
        fi = int(frame_indices[int(i)])
        crop = per_frame_best_crop[fi]
        top_frames.append({
            "frame_index": fi,
            "score": float(per_frame[int(i)]),
            "face_bgr": crop,
            "heatmap_bgr": _fake_heatmap(crop),  # real Grad-CAM
            "bbox": per_frame_best_bbox[fi],
            "n_faces_in_frame": per_frame_n_faces[fi],
        })

    _load_models()
    return {
        "kind": "video",
        "score": agg,
        "verdict": verdict,
        "confidence": conf,
        "per_frame": per_frame,
        "frame_indices": frame_indices,
        "fps_sampled": fps_sample,
        "top_frames": top_frames,
        "tracks": track_results,
        "elapsed_ms": (time.time() - t0) * 1000,
        "model_name": MODEL_NAME,
        "model_version": MODEL_VERSION,
        "meta": {**_MODEL_META,
                 "n_frames": len(per_frame),
                 "n_faces_total": int(len(raw_scores)),
                 "n_tracked_ids": len(per_face_timeline),
                 "track_summaries": {int(k): round(v, 4)
                                     for k, v in track_summaries.items()},
                 "per_face_timeline": {
                     fid: [(fi, round(s, 4), round(r, 4)) for fi, s, r in tl]
                     for fid, tl in per_face_timeline.items()
                 },
                 "avg_faces_per_frame": round(
                     float(np.mean(list(per_frame_n_faces.values()))), 2),
                 "max_face_score": round(float(raw_scores.max()) if len(raw_scores) else 0.0, 4)},
    }


def predict(path: str) -> dict:
    """Route to image or video by extension."""
    ext = os.path.splitext(path)[1].lower()
    if ext in (".mp4", ".mov", ".avi", ".mkv", ".webm"):
        return predict_video(path)
    return predict_image(path)


# --------------- CLI demo ---------------
if __name__ == "__main__":
    import sys, json as _json
    if len(sys.argv) < 2:
        print("usage: python predict.py <file>")
        sys.exit(1)
    r = predict(sys.argv[1])
    # strip numpy arrays for pretty print
    printable = {k: v for k, v in r.items()
                 if not isinstance(v, (np.ndarray,))}
    printable.pop("top_frames", None)
    printable.pop("face_crop_bgr", None)
    printable.pop("heatmap_bgr", None)
    print(_json.dumps(printable, indent=2, default=str))
    print(f"\nverdict: {r['verdict']}  score: {r['score']:.4f}  "
          f"confidence: {r['confidence']:.2f}  elapsed: {r['elapsed_ms']:.0f}ms")


## `forensics.py` — hashing, EXIF, ELA

In [ ]:
"""
Real forensics: perceptual hash, EXIF extraction, basic ELA.

Public API used by app.py:
    compute_phash(file_path) -> str
    extract_exif(file_path)  -> dict
    ela_image(file_path)     -> np.ndarray | None
"""
from __future__ import annotations
import os, subprocess, json
from typing import Optional
from PIL import Image, ImageChops, ImageEnhance, ExifTags


VIDEO_EXTS = (".mp4", ".mov", ".avi", ".mkv", ".webm")


def _is_video(path: str) -> bool:
    return path.lower().endswith(VIDEO_EXTS)


def compute_phash(file_path: str) -> str:
    """
    Perceptual hash. For images, uses a fast difference-hash (dHash) implementation
    that doesn't require the `imagehash` package. For videos, hashes the middle frame.
    Returns a 16-char hex string.
    """
    try:
        if _is_video(file_path):
            img = _video_middle_frame(file_path)
            if img is None:
                return "unavailable"
        else:
            img = Image.open(file_path).convert("L")
        return _dhash(img)
    except Exception:
        return "unavailable"


def _dhash(img: Image.Image, hash_size: int = 8) -> str:
    """Difference hash — 64-bit → 16 hex chars. No numpy needed."""
    img = img.convert("L").resize((hash_size + 1, hash_size), Image.LANCZOS)
    pixels = list(img.getdata())
    bits = []
    for row in range(hash_size):
        for col in range(hash_size):
            left  = pixels[row * (hash_size + 1) + col]
            right = pixels[row * (hash_size + 1) + col + 1]
            bits.append(1 if left > right else 0)
    val = 0
    for b in bits:
        val = (val << 1) | b
    return f"{val:016x}"


def _video_middle_frame(path: str) -> Optional[Image.Image]:
    try:
        import cv2
        cap = cv2.VideoCapture(path)
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, n // 2))
        ok, frame = cap.read()
        cap.release()
        if not ok or frame is None:
            return None
        return Image.fromarray(frame[..., ::-1])
    except Exception:
        return None


def extract_exif(file_path: str) -> dict:
    """
    Return EXIF (image) or ffprobe-derived container/codec info (video).
    """
    if _is_video(file_path):
        return _ffprobe_meta(file_path)
    try:
        img = Image.open(file_path)
        raw = img.getexif()
        out = {}
        for tag_id, val in raw.items():
            tag = ExifTags.TAGS.get(tag_id, str(tag_id))
            # keep serialisable values only
            if isinstance(val, (str, int, float)):
                out[tag] = val
            elif isinstance(val, bytes):
                try:
                    out[tag] = val.decode("utf-8", errors="ignore")[:200]
                except Exception:
                    out[tag] = f"<{len(val)} bytes>"
            else:
                out[tag] = str(val)[:200]
        out["_image_size"]  = f"{img.width}x{img.height}"
        out["_image_mode"]  = img.mode
        out["_image_format"] = img.format or "unknown"
        return out
    except Exception as e:
        return {"error": str(e)}


def _ffprobe_meta(path: str) -> dict:
    try:
        r = subprocess.run(
            ["ffprobe", "-v", "quiet", "-print_format", "json",
             "-show_format", "-show_streams", path],
            capture_output=True, text=True, timeout=15,
        )
        if r.returncode != 0:
            return {"note": "ffprobe not available or failed"}
        j = json.loads(r.stdout)
        fmt = j.get("format", {}); streams = j.get("streams", [])
        v = next((s for s in streams if s.get("codec_type") == "video"), {})
        a = next((s for s in streams if s.get("codec_type") == "audio"), {})
        return {
            "format_name":    fmt.get("format_name"),
            "duration_sec":   float(fmt.get("duration", 0) or 0),
            "size_bytes":     int(fmt.get("size", 0) or 0),
            "bit_rate":       int(fmt.get("bit_rate", 0) or 0),
            "video_codec":    v.get("codec_name"),
            "video_size":     f"{v.get('width', '?')}x{v.get('height', '?')}",
            "video_fps":      v.get("r_frame_rate"),
            "video_pix_fmt":  v.get("pix_fmt"),
            "audio_codec":    a.get("codec_name") if a else None,
            "audio_channels": a.get("channels") if a else None,
        }
    except FileNotFoundError:
        return {"note": "ffprobe (from ffmpeg) not installed"}
    except Exception as e:
        return {"error": str(e)}


def ela_image(file_path: str):
    """
    Error Level Analysis for images. Re-saves at JPEG quality 90 and returns
    an amplified difference — genuine areas fade to near-black, edited/pasted
    regions show up brighter.
    Returns a numpy array (RGB uint8) or None on failure.
    """
    if _is_video(file_path):
        return None
    try:
        import numpy as np
        orig = Image.open(file_path).convert("RGB")
        buf_path = file_path + ".ela_tmp.jpg"
        orig.save(buf_path, "JPEG", quality=90)
        try:
            resaved = Image.open(buf_path).convert("RGB")
            diff = ImageChops.difference(orig, resaved)
            extrema = diff.getextrema()
            max_diff = max((e[1] for e in extrema), default=1) or 1
            scale = 255.0 / max_diff
            diff = ImageEnhance.Brightness(diff).enhance(scale)
            return np.asarray(diff, dtype=np.uint8)
        finally:
            try: os.remove(buf_path)
            except Exception: pass
    except Exception:
        return None


## `report.py` — reportlab PDF evidence report

In [ ]:
"""
Real PDF report using reportlab.
Falls back to a valid minimal-stub PDF if reportlab is missing.

Public API used by app.py:
    generate_report(input_path, verdict, sha256, phash) -> bytes
"""
from __future__ import annotations
import io, os, datetime
from typing import Any


def generate_report(input_path: str, verdict: dict,
                    sha256: str, phash: str) -> bytes:
    try:
        return _real_report(input_path, verdict, sha256, phash)
    except Exception as e:
        return _stub_pdf(f"report generation failed: {e}")


def _real_report(input_path: str, verdict: dict,
                 sha256: str, phash: str) -> bytes:
    from reportlab.lib.pagesizes import LETTER
    from reportlab.lib import colors
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer,
                                     Table, TableStyle, PageBreak, Image as RLImage)
    from reportlab.pdfgen import canvas as rlcanvas

    buf = io.BytesIO()
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="H", fontSize=18, spaceAfter=8,
                              textColor=colors.HexColor("#1e3a8a"),
                              fontName="Helvetica-Bold"))
    styles.add(ParagraphStyle(name="SubH", fontSize=13, spaceBefore=14, spaceAfter=6,
                              textColor=colors.HexColor("#1e3a8a"),
                              fontName="Helvetica-Bold"))
    styles.add(ParagraphStyle(name="Verdict", fontSize=22, alignment=1,
                              fontName="Helvetica-Bold", spaceAfter=10))
    styles.add(ParagraphStyle(name="Small", fontSize=8,
                              textColor=colors.grey))

    doc = SimpleDocTemplate(buf, pagesize=LETTER,
                            leftMargin=0.7 * inch, rightMargin=0.7 * inch,
                            topMargin=0.6 * inch, bottomMargin=0.6 * inch,
                            title="Evidence Analysis Report")

    story = []

    # ---- Header ----
    story.append(Paragraph("🛡  Digital Evidence Analysis Report", styles["H"]))
    story.append(Paragraph("Generated by DeepGuard AI  ·  IEEE AIML-02", styles["Small"]))
    story.append(Spacer(1, 12))

    # ---- Verdict banner ----
    v = verdict.get("verdict", "?")
    score = float(verdict.get("score", 0.0)) * 100.0
    conf  = float(verdict.get("confidence", 0.0)) * 100.0
    color = colors.HexColor("#ef4444") if v == "FAKE" else \
            colors.HexColor("#60a5fa") if v == "NO_FACE" else \
            colors.HexColor("#f59e0b") if v == "INCONCLUSIVE" else \
            colors.HexColor("#10b981")
    icon  = "⚠" if v == "FAKE" else "?" if v in ("INCONCLUSIVE", "NO_FACE") else "✓"
    label = "MANIPULATION SUSPECTED" if v == "FAKE" else \
            "NO SUITABLE FACE" if v == "NO_FACE" else \
            "INCONCLUSIVE" if v == "INCONCLUSIVE" else "LIKELY AUTHENTIC"

    banner = Table([[f"{icon}   {label}"]], colWidths=[7 * inch])
    banner.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), color),
        ("TEXTCOLOR",  (0, 0), (-1, -1), colors.white),
        ("FONTNAME",   (0, 0), (-1, -1), "Helvetica-Bold"),
        ("FONTSIZE",   (0, 0), (-1, -1), 20),
        ("ALIGN",      (0, 0), (-1, -1), "CENTER"),
        ("TOPPADDING", (0, 0), (-1, -1), 14),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 14),
        ("ROUNDEDCORNERS", [6, 6, 6, 6]),
    ]))
    story.append(banner)
    story.append(Spacer(1, 6))
    story.append(Paragraph(
        f"<para align=center><b>Fake probability: {score:.2f}%  ·  "
        f"Confidence: {conf:.2f}%  ·  Inference: {verdict.get('elapsed_ms', 0):.0f} ms</b></para>",
        styles["Normal"],
    ))
    story.append(Spacer(1, 14))

    # ---- Exhibit info ----
    story.append(Paragraph("Exhibit Information", styles["SubH"]))
    fname = os.path.basename(input_path)
    fsize = os.path.getsize(input_path) if os.path.exists(input_path) else 0
    exhibit_rows = [
        ["Filename",       fname],
        ["File size",      f"{fsize:,} bytes"],
        ["Kind",           verdict.get("kind", "?").upper()],
        ["Analyzed at",    datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S %Z")],
        ["SHA-256",        _wrap(sha256)],
        ["Perceptual hash", phash or "—"],
    ]
    story.append(_kv_table(exhibit_rows, TableStyle, colors))

    # ---- Model info ----
    story.append(Paragraph("Detection Model", styles["SubH"]))
    model_rows = [
        ["Model",   verdict.get("model_name", "?")],
        ["Version", verdict.get("model_version", "?")],
        ["Verdict", v],
        ["Fake probability", f"{score:.4f}"],
        ["Confidence",       f"{conf:.4f}"],
    ]
    story.append(_kv_table(model_rows, TableStyle, colors))

    # ---- Video timeline ----
    if verdict.get("kind") == "video":
        story.append(Paragraph("Per-Frame Timeline (video)", styles["SubH"]))
        img_bytes = _timeline_png(verdict)
        if img_bytes:
            story.append(RLImage(io.BytesIO(img_bytes), width=6.6 * inch, height=2.2 * inch))
        else:
            story.append(Paragraph("<i>Timeline chart unavailable.</i>", styles["Normal"]))

        tf = verdict.get("top_frames", [])
        if tf:
            story.append(Spacer(1, 8))
            story.append(Paragraph(
                f"Top {len(tf)} most-suspicious sampled frames:", styles["Normal"]))
            top_rows = [["Rank", "Frame index", "Fake score"]]
            for i, f in enumerate(tf, 1):
                top_rows.append([str(i), str(f.get("frame_index", "?")),
                                 f"{float(f.get('score', 0)) * 100:.1f}%"])
            t = Table(top_rows, colWidths=[0.7 * inch, 2 * inch, 2 * inch])
            t.setStyle(TableStyle([
                ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1e3a8a")),
                ("TEXTCOLOR",  (0, 0), (-1, 0), colors.white),
                ("FONTNAME",   (0, 0), (-1, 0), "Helvetica-Bold"),
                ("FONTSIZE",   (0, 0), (-1, -1), 9),
                ("ALIGN",      (0, 0), (-1, -1), "CENTER"),
            ]))
            story.append(t)

    tracks = verdict.get("tracks") or []
    if tracks:
        story.append(Spacer(1, 10))
        story.append(Paragraph("Per-Person Track Results", styles["SubH"]))
        rows = [["Track", "Observations", "Score", "Verdict"]]
        for track in tracks:
            rows.append([str(track.get("face_id", "?")),
                         str(track.get("observations", 0)),
                         f"{float(track.get('score', 0))*100:.1f}%",
                         track.get("verdict", "INCONCLUSIVE")])
        story.append(_kv_table(rows, TableStyle, colors))

    story.append(Spacer(1, 12))
    story.append(Paragraph("Interpretation and Limitations", styles["SubH"]))
    story.append(Paragraph(
        "This automated result is a screening aid, not proof of authenticity or manipulation. "
        "Scores may be unreliable for unseen manipulation methods, heavy compression, blur, "
        "small faces, screenshots, or re-encoded media. EXIF absence and compression anomalies "
        "are supporting indicators only. Preserve the original exhibit and obtain qualified "
        "human forensic review before legal or judicial reliance.", styles["Small"]))
    # ---- Analyst notes ----
    story.append(Spacer(1, 14))
    story.append(Paragraph("Analyst Notes", styles["SubH"]))
    story.append(Paragraph("<i>Space for handwritten annotations, case number, "
                           "and analyst signature below.</i>", styles["Small"]))
    notes = Table([[" "]] * 4, colWidths=[7 * inch])
    notes.setStyle(TableStyle([
        ("BOX",  (0, 0), (-1, -1), 0.5, colors.grey),
        ("LINEBELOW", (0, 0), (-1, -2), 0.4, colors.grey),
        ("ROWHEIGHTS", (0, 0), (-1, -1), 0.4 * inch),
    ]))
    story.append(notes)

    story.append(Spacer(1, 20))
    story.append(Paragraph(
        "This report was generated by an automated forensic assistant. "
        "Findings should be reviewed by a qualified digital-forensics analyst "
        "before being used as evidence.",
        styles["Small"],
    ))

    doc.build(story, onFirstPage=_footer, onLaterPages=_footer)
    return buf.getvalue()


def _kv_table(rows, TableStyle, colors):
    from reportlab.lib.units import inch
    from reportlab.platypus import Table
    t = Table(rows, colWidths=[1.7 * inch, 4.8 * inch])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#eef2ff")),
        ("TEXTCOLOR",  (0, 0), (0, -1), colors.HexColor("#1e3a8a")),
        ("FONTNAME",   (0, 0), (0, -1), "Helvetica-Bold"),
        ("FONTNAME",   (1, 0), (1, -1), "Helvetica"),
        ("FONTSIZE",   (0, 0), (-1, -1), 9),
        ("VALIGN",     (0, 0), (-1, -1), "TOP"),
        ("GRID",       (0, 0), (-1, -1), 0.3, colors.HexColor("#d1d5db")),
        ("LEFTPADDING",  (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("TOPPADDING",   (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING",(0, 0), (-1, -1), 5),
    ]))
    return t


def _wrap(s, per=48):
    """Insert soft breaks so long hashes don't overflow."""
    return " ".join(s[i:i + per] for i in range(0, len(s), per)) if s else ""


def _footer(canvas, doc):
    from reportlab.lib import colors as rc
    canvas.saveState()
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(rc.grey)
    canvas.drawString(0.7 * 72, 0.4 * 72,
                      "DeepGuard AI · Digital Evidence Report")
    canvas.drawRightString(8.3 * 72, 0.4 * 72, f"Page {doc.page}")
    canvas.restoreState()


def _timeline_png(verdict: dict) -> bytes | None:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        import numpy as np
        xs = np.array(verdict.get("frame_indices", []))
        ys = np.array(verdict.get("per_frame", []))
        if len(xs) == 0:
            return None
        fig, ax = plt.subplots(figsize=(6.6, 2.2))
        ax.plot(xs, ys, color="#3b82f6", lw=1.5)
        ax.fill_between(xs, ys, 0.35, where=(ys > 0.35), color="#fca5a5", alpha=0.5)
        ax.fill_between(xs, ys, 0.35, where=(ys <= 0.35), color="#a7f3d0", alpha=0.5)
        ax.axhline(0.35, color="#f59e0b", lw=1.0, ls="--")
        ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel("Frame index"); ax.set_ylabel("Fake prob.")
        ax.set_title(f"Overall screening score: {verdict.get('score', 0)*100:.1f}%")
        fig.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=110)
        plt.close(fig)
        return buf.getvalue()
    except Exception:
        return None


def _stub_pdf(msg: str) -> bytes:
    body = (
        b"BT /F1 14 Tf 72 720 Td (Evidence Report) Tj ET\n"
        b"BT /F1 10 Tf 72 690 Td (" + msg.encode("latin-1", "ignore") + b") Tj ET\n"
    )
    return (
        b"%PDF-1.4\n"
        b"1 0 obj<</Type/Catalog/Pages 2 0 R>>endobj\n"
        b"2 0 obj<</Type/Pages/Kids[3 0 R]/Count 1>>endobj\n"
        b"3 0 obj<</Type/Page/MediaBox[0 0 612 792]/Parent 2 0 R"
        b"/Resources<</Font<</F1 4 0 R>>>>/Contents 5 0 R>>endobj\n"
        b"4 0 obj<</Type/Font/Subtype/Type1/BaseFont/Helvetica>>endobj\n"
        b"5 0 obj<</Length " + str(len(body)).encode() + b">>stream\n" + body +
        b"endstream endobj\n"
        b"xref 0 6 trailer<</Size 6/Root 1 0 R>>startxref 0 %%EOF\n"
    )


## `sanity_test.py` — OOD verification harness

In [ ]:
"""
Sanity test: download a small set of out-of-distribution files and run
them through predict.py. Prints a table of expected vs actual verdicts.

Establishes a baseline BEFORE any retraining, so we know if the retrain
actually improved things.

Usage:
    python sanity_test.py
"""
import os, sys, time, urllib.request, urllib.error, json, ssl
from pathlib import Path

CACHE = Path(r"C:\dl\deepfake\data\sanity")
CACHE.mkdir(parents=True, exist_ok=True)

# Test cases: (source_url, filename, expected, description)
# Real: deepface's canonical demo photos (raw GitHub, reliable)
# Fake: NVIDIA StyleGAN2-ADA official generated samples (raw GitHub, reliable)
TESTS = [
    # Real face photos — Wikimedia raw file URLs (Special:FilePath resolves any format)
    ("https://commons.wikimedia.org/wiki/Special:FilePath/Barack_Obama_official_White_House_portrait_June_2012.jpg?width=512",
     "real_obama.jpg", "REAL", "Real photo (Obama, Wikimedia)"),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/Angela_Merkel_2019_cropped.jpg?width=512",
     "real_merkel.jpg", "REAL", "Real photo (Merkel, Wikimedia)"),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/Emma_Watson_2013.jpg?width=512",
     "real_emma.jpg", "REAL", "Real photo (Emma Watson, Wikimedia)"),
    # Fake face photos — deepfake / synthetic image samples
    ("https://raw.githubusercontent.com/NVlabs/stylegan2-ada-pytorch/main/docs/stylegan2-ada-teaser-1024x252.png",
     "fake_stylegan_grid.png", "FAKE", "StyleGAN2 teaser (NVIDIA)"),
    ("https://raw.githubusercontent.com/NVlabs/stylegan3/main/docs/stylegan3-teaser-1920x1006.png",
     "fake_stylegan3.png", "FAKE", "StyleGAN3 teaser (NVIDIA)"),
]


def _download(url: str, path: Path, force: bool = False) -> bool:
    if path.exists() and not force:
        return True
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "sanity-test/1.0"})
        ctx = ssl.create_default_context()
        with urllib.request.urlopen(req, timeout=30, context=ctx) as r:
            path.write_bytes(r.read())
        return True
    except Exception as e:
        print(f"  DOWNLOAD FAILED: {url}  err={e}", file=sys.stderr)
        return False


def _colored(text: str, ok: bool) -> str:
    return f"\033[92m{text}\033[0m" if ok else f"\033[91m{text}\033[0m"


def main():
    from predict import predict

    print("=" * 88)
    print(f"SANITY TEST — {len(TESTS)} out-of-distribution files")
    print("=" * 88)

    # Download all
    print("\n[1/2] Downloading test files...")
    for url, name, _, desc in TESTS:
        path = CACHE / name
        # tpdne returns fresh face each hit — force refresh
        force = "tpdne" in name
        ok = _download(url, path, force=force)
        print(f"   {'OK' if ok else 'FAIL':4s}  {name:20s}  ({desc})")

    print("\n[2/2] Running predictions...")
    print()
    print(f"  {'FILE':<20s}  {'EXPECTED':<10s}  {'ACTUAL':<12s}  {'SCORE':<7s}  {'CONF':<6s}  {'HIT?'}")
    print(f"  {'-'*20}  {'-'*10}  {'-'*12}  {'-'*7}  {'-'*6}  {'-'*4}")

    results = []
    hits = 0
    for _, name, expected, desc in TESTS:
        path = CACHE / name
        if not path.exists():
            print(f"  {name:<20s}  {expected:<10s}  {'MISSING':<12s}  -        -       ✗")
            continue
        t = time.time()
        try:
            r = predict(str(path))
        except Exception as e:
            print(f"  {name:<20s}  {expected:<10s}  ERR: {str(e)[:40]}")
            continue
        actual = r["verdict"]
        score  = r["score"]
        conf   = r["confidence"]
        hit    = (actual == expected) or (expected == "REAL" and score < 0.5) or (expected == "FAKE" and score > 0.5)
        results.append({"file": name, "expected": expected, "actual": actual,
                        "score": score, "conf": conf, "hit": bool(hit),
                        "elapsed_ms": r["elapsed_ms"], "desc": desc})
        if hit: hits += 1
        mark = "✓" if hit else "✗"
        print(f"  {name:<20s}  {expected:<10s}  {actual:<12s}  {score:.4f}   {conf:.2f}    {mark}")

    print()
    print("=" * 88)
    print(f"  HITS: {hits} / {len(results)}  "
          f"({100.0 * hits / max(1, len(results)):.0f}%)")
    print("=" * 88)

    Path(r"C:\dl\deepfake\sanity_result.json").write_text(
        json.dumps({"hits": hits, "total": len(results),
                    "results": results}, indent=2))
    print(f"\nSaved sanity_result.json")


if __name__ == "__main__":
    main()


## `challenge_eval.py` — external difficult-case evaluation harness

In [ ]:
"""Small external challenge-set evaluator for DeepGuard.

Directory layout:
    challenge_set/
      real/          genuine photos/videos
      fake/          face swaps, reenactment, synthetic media
      inconclusive/  deliberately blurry/dark/tiny-face samples
      no_face/       documents, rooms, landscapes

Files are evaluated independently of the training datasets. Results are saved
as challenge_result.json for demo reproducibility.
"""
from __future__ import annotations

import argparse
import json
from collections import Counter, defaultdict
from pathlib import Path

from predict import predict


EXPECTED = {
    "real": "REAL",
    "fake": "FAKE",
    "inconclusive": "INCONCLUSIVE",
    "no_face": "NO_FACE",
}
MEDIA_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".mp4", ".mov", ".avi", ".mkv", ".webm"}


def iter_cases(root: Path):
    for category, expected in EXPECTED.items():
        folder = root / category
        if not folder.exists():
            continue
        for path in sorted(folder.rglob("*")):
            if path.is_file() and path.suffix.lower() in MEDIA_EXTS:
                yield category, expected, path


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("root", nargs="?", default="challenge_set")
    parser.add_argument("--output", default="challenge_result.json")
    args = parser.parse_args()

    root = Path(args.root)
    rows = []
    confusion = Counter()
    category_stats = defaultdict(lambda: {"correct": 0, "total": 0})

    for category, expected, path in iter_cases(root):
        try:
            result = predict(str(path))
            predicted = result["verdict"]
            row = {
                "path": str(path),
                "category": category,
                "expected": expected,
                "predicted": predicted,
                "score": round(float(result["score"]), 6),
                "confidence": round(float(result["confidence"]), 6),
                "correct": predicted == expected,
                "error": None,
            }
        except Exception as exc:
            predicted = "ERROR"
            row = {
                "path": str(path), "category": category, "expected": expected,
                "predicted": predicted, "score": None, "confidence": None,
                "correct": False, "error": f"{type(exc).__name__}: {exc}",
            }
        rows.append(row)
        confusion[(expected, predicted)] += 1
        category_stats[category]["total"] += 1
        category_stats[category]["correct"] += int(row["correct"])
        print(f"{category:12s} expected={expected:12s} got={predicted:12s} {path.name}")

    total = len(rows)
    correct = sum(int(row["correct"]) for row in rows)
    summary = {
        "root": str(root.resolve()),
        "n_cases": total,
        "accuracy": (correct / total) if total else None,
        "by_category": {
            key: {**value, "accuracy": value["correct"] / value["total"]}
            for key, value in category_stats.items() if value["total"]
        },
        "confusion": {
            f"{expected}->{predicted}": count
            for (expected, predicted), count in sorted(confusion.items())
        },
        "cases": rows,
        "warning": "Internal screening metric only; not a judicial validation study.",
    }
    Path(args.output).write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps({k: v for k, v in summary.items() if k != "cases"}, indent=2))
    return 0 if total else 2


if __name__ == "__main__":
    raise SystemExit(main())


## `video_test.py` — end-to-end video pipeline smoke test

In [ ]:
"""
End-to-end video test: download a small video, run through predict_video,
print per-frame timeline + verdict. Proves the full video pipeline works.
"""
import os, sys, time, urllib.request
from pathlib import Path
sys.path.insert(0, r"C:\dl\deepfake")

CACHE = Path(r"C:\dl\deepfake\data\video_test")
CACHE.mkdir(parents=True, exist_ok=True)

# Small, freely-hostable video (a well-known Blender open movie clip)
VIDEO_URL = "https://sample-videos.com/video321/mp4/720/big_buck_bunny_720p_1mb.mp4"
LOCAL     = CACHE / "sample.mp4"

def dl(url, path):
    if path.exists() and path.stat().st_size > 0:
        return
    print(f"downloading {url}...")
    req = urllib.request.Request(url, headers={"User-Agent": "video-test/1.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        path.write_bytes(r.read())

def main():
    from predict import predict_video
    try:
        dl(VIDEO_URL, LOCAL)
    except Exception as e:
        print(f"download failed: {e}")
        # fall back to legion's test video if present
        legion = Path(r"sample.mp4")
        if legion.exists():
            LOCAL.write_bytes(legion.read_bytes())
            print("using legion's sample.mp4 instead")
        else:
            print("no video available"); sys.exit(1)

    print(f"file: {LOCAL}  size={LOCAL.stat().st_size/1024:.0f} KB")
    t = time.time()
    r = predict_video(str(LOCAL), fps_sample=2.0)
    elapsed = time.time() - t
    print()
    print("=" * 70)
    print(f" VIDEO RESULT")
    print("=" * 70)
    print(f"  verdict         : {r['verdict']}")
    print(f"  aggregate score : {r['score']:.4f}  (median of per-frame)")
    print(f"  confidence      : {r['confidence']:.2f}")
    print(f"  frames sampled  : {len(r['per_frame'])}")
    print(f"  fps sampled     : {r['fps_sampled']}")
    print(f"  wall time       : {elapsed:.1f}s  ({r['elapsed_ms']:.0f}ms measured)")
    print(f"  top-5 suspicious frames:")
    for i, tf in enumerate(r['top_frames'], 1):
        print(f"    #{i}  frame {tf['frame_index']:>5d}  score={tf['score']:.3f}")
    print()
    print("  first 20 per-frame scores:")
    for i, s in enumerate(r['per_frame'][:20]):
        bar = "█" * int(s * 40)
        print(f"    frame {r['frame_indices'][i]:>4d}: {s:.3f}  {bar}")
    print("=" * 70)

if __name__ == "__main__":
    main()
